# Timings between POIs

See KNP timings for reference: https://www.sanparks.org/parks/kruger/get_there/KNP_distances_beween_camps.pdf


In [1]:
import pandas as pd
import geopandas
import matplotlib.pyplot as plt

from pyproj import Geod
from shapely.geometry import MultiPoint
from shapely.geometry import Point, LineString, Polygon
import shapely.wkt

from shapely.geometry import shape

import contextily as ctx

import geojson
import datetime
import json

In [2]:
with open('./resources/pois.json') as f:
    d = json.load(f)

In [3]:
x_names = []
for origin in d:

    if origin['Type'] in ['Rest camp', 'Gate']:
        x_names.append(origin['Name'])

# x-axis names are recveresd in KNP chart
x_names = list(reversed(sorted(x_names)))
y_names = sorted(x_names)

In [4]:
head = [''] + list(x_names)
matrix = []

for origin_name in y_names:

    # find poi in json
    origin = None
    for o in d:
        if (o['Name'] == origin_name):
            origin = o
            break;

    #print(origin)
    timings = [origin['Name']]


    for n in x_names:
        found = False

        for dest in origin['Destinations']:
            if n == dest['Name']:
                found = True

                km = str(int(round(dest['Length']/1000, 0)))
                h  = datetime.timedelta(seconds=int(dest['Duration']))
                h = ':'.join(str(h).split(':')[:2])

                timings.append(km + "km\n"  + h)

                break

        if not found:
            timings.append(None)

    matrix.append(timings)

In [5]:
from IPython.display import HTML, display
import tabulate
table = [head] + matrix
display(HTML(tabulate.tabulate(table, tablefmt='html')))

,Skukuza,Shingwedzi,Satara,Punda Maria Gate,Punda Maria,Pretoriuskop,Phalaborwa Gate,Phabeni Gate,Pafuri Gate,Orpen Gate,Orpen,Olifants,Numbi Gate,Mopani,Malelane Gate,Malelane,Lower Sabie,Letaba,Kruger Gate,Crocodile Bridge Gate,Crocodile Bridge,Berg-en-Dal,Balule
Balule,141km 2:50,140km 3:10,50km 0:59,211km 4:15,210km 4:15,189km 3:47,82km 1:43,179km 3:36,247km 4:58,92km 2:09,92km 2:09,13km 0:17,193km 3:58,84km 1:44,204km 4:06,201km 4:03,147km 2:58,35km 0:43,153km 3:04,178km 3:37,178km 3:37,207km 4:12,
Berg-en-Dal,66km 1:22,334km 7:06,159km 3:14,405km 8:10,404km 8:11,59km 1:14,277km 5:39,82km 1:46,441km 8:53,176km 3:55,176km 3:55,214km 4:21,67km 1:24,278km 5:40,16km 0:22,13km 0:19,74km 1:44,229km 4:39,78km 1:36,63km 1:31,63km 1:31,,204km 4:10
Crocodile Bridge,63km 1:23,305km 6:31,130km 2:40,376km 7:36,375km 7:36,91km 2:02,248km 5:04,101km 2:08,412km 8:19,159km 3:35,159km 3:35,185km 3:46,99km 2:06,249km 5:05,60km 1:25,57km 1:22,31km 0:40,200km 4:04,74km 1:36,0km 0:00,,63km 1:31,174km 3:35
Crocodile Bridge Gate,63km 1:23,305km 6:31,130km 2:40,376km 7:36,375km 7:36,91km 2:02,248km 5:04,101km 2:08,412km 8:19,159km 3:35,159km 3:35,185km 3:46,99km 2:06,249km 5:05,60km 1:25,57km 1:22,31km 0:40,200km 4:04,74km 1:36,,0km 0:00,63km 1:31,174km 3:35
Kruger Gate,12km 0:14,280km 5:58,105km 2:06,351km 7:02,350km 7:03,44km 0:56,223km 4:31,29km 0:35,387km 7:45,122km 2:47,122km 2:47,160km 3:13,43km 0:56,224km 4:32,75km 1:30,72km 1:27,56km 1:08,175km 3:31,,74km 1:36,74km 1:36,78km 1:36,149km 3:02
Letaba,164km 3:17,106km 2:29,72km 1:26,177km 3:33,177km 3:33,212km 4:14,49km 1:01,202km 4:03,213km 4:16,115km 2:36,115km 2:36,31km 0:39,216km 4:25,50km 1:02,227km 4:33,224km 4:30,170km 3:25,,175km 3:31,200km 4:04,200km 4:04,229km 4:39,35km 0:43
Lower Sabie,45km 0:55,275km 5:52,100km 2:00,346km 6:56,345km 6:56,88km 1:45,217km 4:25,83km 1:41,382km 7:39,129km 2:56,129km 2:56,155km 3:07,94km 1:52,219km 4:26,71km 1:38,68km 1:35,,170km 3:25,56km 1:08,31km 0:40,31km 0:40,74km 1:44,144km 2:56
Malelane,61km 1:13,329km 6:57,154km 3:05,400km 8:02,399km 8:02,56km 1:07,272km 5:30,79km 1:38,436km 8:44,171km 3:46,171km 3:46,209km 4:12,64km 1:17,273km 5:31,3km 0:03,,68km 1:35,224km 4:30,72km 1:27,57km 1:22,57km 1:22,13km 0:19,198km 4:01
Malelane Gate,64km 1:16,332km 7:00,157km 3:09,403km 8:05,402km 8:05,58km 1:10,274km 5:33,82km 1:41,438km 8:47,174km 3:49,174km 3:49,212km 4:15,66km 1:20,275km 5:34,,3km 0:03,71km 1:38,227km 4:33,75km 1:30,60km 1:25,60km 1:25,16km 0:22,201km 4:04
Mopani,212km 4:18,63km 1:17,121km 2:27,130km 2:36,129km 2:36,260km 5:16,72km 1:37,251km 5:04,166km 3:19,164km 3:37,164km 3:37,81km 1:41,264km 5:26,,275km 5:34,273km 5:31,219km 4:26,50km 1:02,224km 4:32,249km 5:05,249km 5:05,278km 5:40,84km 1:44
